[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Virtual Environments


## What you will be able to do

Make a virtual environment, look inside it, and install packages into it without activating
anything. Show why a project needs one, with two versions of one package that cannot share an
environment, tell which environment a command uses, and rebuild an environment rather than moving
it.


## The idea

### The problem

A tutorial from 2021 sorts the release tags of a package, one of them called `latest`, with
`packaging.version.parse`. On the computer it was written on, it worked. On yours, it stops with
`InvalidVersion: Invalid version: 'latest'`. Nothing is wrong with what you typed: the tutorial was
written against `packaging` 21.3, whose `parse` accepted any text, and the `packaging` in your Python
is a later release, which accepts only versions written the standard way.

Installing `packaging` 21.3 makes the tutorial work, and breaks the project you worked on yesterday,
which calls `canonicalize_license_expression`, a function 21.3 does not have. One Python holds one
version of each package, so the two programs cannot both run in it. The **Environments and pip**
notebook in the **Python from the Start** guide showed the commands that avoid this. This notebook
opens the folder those commands make, and uses it the way tools do.

### What a virtual environment is

> A **virtual environment** is a folder that works as a Python installation of its own.
> `python -m venv .venv` makes one: a `bin` folder, `Scripts` on Windows, holding a `python`; a `lib`
> folder with the environment's own `site-packages`; and a file, `pyvenv.cfg`, that names the Python
> the environment was made from. Running the environment's `python` runs that Python with the
> environment's packages in place of its own, so two environments can hold two versions of one
> package. **Activating** an environment changes only the shell it runs in: it puts the environment's
> `bin` first on `PATH`, so that typing `python` finds the environment's copy. Python's documentation
> calls an environment disposable, rebuilt rather than moved or copied, and never checked into
> source control.

### Why it works that way

- **`pyvenv.cfg` is how Python knows.** The environment's `python` is a link to, or a copy of, the
  Python it was made from. When it starts, it finds `pyvenv.cfg` one folder above itself, sets
  `sys.prefix` to the environment's folder, and takes its packages from that folder's
  `site-packages`.
- **The `python` you run decides the environment.** `.venv/bin/python` runs inside the environment
  whether or not anything was activated, which is how a script or a notebook uses one.
- **An environment holds one version of each package.** Installing another version replaces the
  first, so two projects that need two versions need two environments.
- **An environment sees only its own packages.** It sees the packages of the Python it came from only
  if it was made with `--system-site-packages`.
- **An environment is tied to the folder it was made in.** A script that pip installs, such as
  `pytest`, begins with a line naming the environment's `python` by its full path, and so does the
  `activate` script, so a moved environment's scripts point at a folder that is gone.
- **An environment is cheap to rebuild.** A list of what it held is enough, which is the subject of
  the **Requirements and Pinning** notebook, and the folder itself holds nothing worth keeping.

### Where this shows up

Python's documentation for `venv` describes an environment as disposable, not movable or copyable,
and not checked into source control, and since Python 3.13 `python -m venv` writes a `.gitignore`
into a new environment, so that Git leaves it out without being told. pip's documentation shows
`python -m pip --python .venv install`, the form this notebook uses, for an environment made without
pip. On Ubuntu, `ensurepip`, the module `venv` runs to put pip into a new environment, comes in a
separate package, `python3-venv`, and Colab's published list of installed packages does not include
it. Without it, `python -m venv .venv` stops with an error that begins "The virtual environment was
not created successfully because ensurepip is not available". So every environment in this notebook
is made with `--without-pip`, which does not run `ensurepip`, and packages go in with the notebook's
own pip.

### What this notebook covers

- A new environment, and what is inside it
- The environment's `python`, run without activating anything
- Packages installed into an environment with the notebook's own pip and `--python`
- The tutorial that broke: two versions of `packaging`, in two environments
- What activating changes, and what it leaves alone
- `--system-site-packages`, an environment that sees the packages around it
- A project's environment, from nothing to its tests passing inside it
- Four errors: an environment with no pip, a package installed into another environment, an
  environment's own `pytest` that cannot find the project, and an environment that moved

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import subprocess
import sys
import tempfile
from pathlib import Path

folder = Path(tempfile.mkdtemp())
subprocess.run([sys.executable, "-m", "venv", "--without-pip", folder / "env"], check=True)

check = "import sys; print('inside an environment:', sys.prefix != sys.base_prefix)"
subprocess.run([sys.executable, "-c", check])
subprocess.run([folder / "env" / "bin" / "python", "-c", check])
print(sorted(path.name for path in (folder / "env").iterdir()))
```

```
inside an environment: False
inside an environment: True
['.gitignore', 'bin', 'include', 'lib', 'pyvenv.cfg']
```

The same check, run by two Pythons: the one running the program, and the `python` inside a folder
that `venv` had made a moment before. Nothing was activated. Running the environment's `python` was
enough to be inside the environment, and the folder holds everything that makes it one.


## Setup

Six imports, and two functions that run commands.

- `subprocess` runs `venv`, pip and the environments' Pythons as programs of their own
- `sys` names the notebook's own Python, which makes every environment and runs pip
- `os` sets `NO_COLOR`, `PYTHONDONTWRITEBYTECODE` and `COLUMNS` for those programs, as the **Your
  First Test** notebook did
- `re` takes the time a test run took out of pytest's report
- `Path` builds the paths of the environments, and reads the files inside them
- `shutil` moves and removes environments, and removes the scratch folder at the end

`run` runs a command in the project's folder, `scratch/stations`, and returns its exit code and what
it printed, with the folder's full path taken out, since it differs on every computer. `pip` runs the
notebook's own pip for the Python in an environment, which the section on installing explains.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("scratch/stations")
(PROJECT / "tests").mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun
os.environ["COLUMNS"] = "80"                  # and print reports 80 characters wide


def run(*command, folder=PROJECT):
    """Run a command in a folder, and return its exit code and what it printed, less this computer's paths."""
    finished = subprocess.run([str(part) for part in command], cwd=folder, capture_output=True, text=True)
    printed = (finished.stdout + finished.stderr).replace(f"{Path(folder).resolve()}/", "")
    return finished.returncode, re.sub(r" in \d+\.\d+s\b", "", printed).rstrip()


def pip(environment, *arguments):
    """Run this notebook's pip for the Python in an environment: python -m pip --python ENVIRONMENT ..."""
    return run(sys.executable, "-m", "pip", "--python", environment, "--disable-pip-version-check", *arguments)


print("ready:", PROJECT)


ready: scratch/stations


## Worked examples

### A new environment, and what is inside it

`python -m venv --without-pip old-tutorial` makes an environment in a folder called `old-tutorial`,
inside the project's folder:


In [2]:
code, printed = run(sys.executable, "-m", "venv", "--without-pip", "old-tutorial")

print("exit code:", code, "| printed:", repr(printed))
print(sorted(path.name for path in (PROJECT / "old-tutorial").iterdir()))


exit code: 0 | printed: ''
['.gitignore', 'bin', 'include', 'lib', 'pyvenv.cfg']


`venv` printed nothing, and made a folder with four things in it, and a fifth from Python 3.13 on:

| In the folder | What it is |
|---|---|
| `bin` | the environment's `python`, and later the scripts that packages install, such as `pytest` |
| `lib` | the environment's own `site-packages`, where pip puts the packages |
| `include` | a place for the C headers some packages need to build |
| `pyvenv.cfg` | the file that makes the folder an environment |
| `.gitignore` | a line, `*`, that keeps the whole folder out of a Git repository |

Here are the keys in `pyvenv.cfg`, and the packages in the environment's `site-packages`, which the
environment's own `python` finds:


In [3]:
config = (PROJECT / "old-tutorial" / "pyvenv.cfg").read_text()
print([line.split(" = ")[0] for line in config.splitlines()])
print([line for line in config.splitlines() if line.startswith("include-system-site-packages")])

listing = "import os, sysconfig; print(os.listdir(sysconfig.get_paths()['purelib']))"
code, printed = run("old-tutorial/bin/python", "-c", listing)
print("packages in the environment:", printed)


['home', 'include-system-site-packages', 'version', 'executable', 'command']
['include-system-site-packages = false']
packages in the environment: []


`home` and `executable` name the Python the environment was made from, `version` its version, and
`command` the command that made it; all four hold full paths or details of this computer, so the cell
prints only their names. `include-system-site-packages = false` keeps the environment from seeing the
packages of that Python. The environment's `site-packages` is empty.

### The environment's python, run without activating anything

The same one-line check, run by the notebook's Python and by the environment's:


In [4]:
check = "import sys; print('inside an environment:', sys.prefix != sys.base_prefix)"

for python in [sys.executable, "old-tutorial/bin/python"]:
    code, printed = run(python, "-c", check)
    name = "the notebook's Python" if python == sys.executable else python
    print(f"{name:<25} {printed}")


the notebook's Python     inside an environment: False
old-tutorial/bin/python   inside an environment: True


Nothing was activated. `old-tutorial/bin/python` is a link to the Python the environment was made
from, and when it starts, it finds `pyvenv.cfg` one folder up and sets `sys.prefix` to the
environment's folder, while `sys.base_prefix` stays the Python it came from. The two differ only
inside an environment, as the **Environments and pip** notebook showed. Running the environment's
`python` by its path is all it takes to use the environment, which is how the rest of this notebook
uses one.

### Packages installed with the notebook's own pip

An environment made with `--without-pip` has no pip of its own. `python -m pip --python old-tutorial`
runs the notebook's pip, and tells it to install for the Python in `old-tutorial`, which is what the
`pip` function does. Here the environment gets `packaging` 21.3, the version the tutorial was written
against, and `pyparsing`, which that version of `packaging` needs:


In [5]:
code, printed = pip("old-tutorial", "install", "-q", "packaging==21.3", "pyparsing==3.3.2")
print("exit code:", code)

code, printed = pip("old-tutorial", "freeze")
print(printed)


exit code: 0
packaging==21.3
pyparsing==3.3.2


`-q` kept pip quiet, and `freeze` lists what the environment holds: the two packages installed, and
nothing else. On your own computer, `python -m venv .venv` puts pip into the environment, and
`.venv/bin/python -m pip install packaging==21.3` does the same job with the environment's own pip.

### The tutorial that broke

Here are the tutorial and yesterday's project, as two scripts:


In [6]:
%%writefile scratch/stations/tutorial.py
"""A tutorial from 2021: the release tags of a package, in order."""

from packaging.version import parse

tags = ["2.10", "2.9", "2.10rc1", "latest"]
print(sorted(tags, key=parse))


Writing scratch/stations/tutorial.py


In [7]:
%%writefile scratch/stations/current.py
"""Yesterday's project: a license, written the way packages record one today."""

from packaging.licenses import canonicalize_license_expression

print(canonicalize_license_expression("mit or apache-2.0"))


Writing scratch/stations/current.py


Both scripts, run by the notebook's Python, and then by the environment's. The cell prints the exit
code and the last line each script printed, which for a script that failed is its exception:


In [8]:
for python in [sys.executable, "old-tutorial/bin/python"]:
    print("the notebook's Python:" if python == sys.executable else f"{python}:")
    for script in ["tutorial.py", "current.py"]:
        code, printed = run(python, script)
        print(f"  {script:<12} exit code {code}: {printed.splitlines()[-1]}")


the notebook's Python:
  tutorial.py  exit code 1: packaging.version.InvalidVersion: Invalid version: 'latest'
  current.py   exit code 0: MIT OR Apache-2.0
old-tutorial/bin/python:
  tutorial.py  exit code 0: ['latest', '2.9', '2.10rc1', '2.10']
  current.py   exit code 1: ModuleNotFoundError: No module named 'packaging.licenses'


The notebook's Python has a later `packaging`, which raises `InvalidVersion` for `latest`, and which
has `packaging.licenses`. The environment has 21.3, where `parse` accepted `latest` and sorted it
before every real version, and where `packaging.licenses` does not exist yet. Neither Python runs
both scripts. A second environment, for the current project, holds a current `packaging`:


In [9]:
run(sys.executable, "-m", "venv", "--without-pip", "current")
code, printed = pip("current", "install", "-q", "packaging==26.3")
print("exit code:", code)

for python, script in [("old-tutorial/bin/python", "tutorial.py"), ("current/bin/python", "current.py")]:
    code, printed = run(python, script)
    print(f"{python:<24} {script:<12} exit code {code}: {printed.splitlines()[-1]}")


exit code: 0
old-tutorial/bin/python  tutorial.py  exit code 0: ['latest', '2.9', '2.10rc1', '2.10']
current/bin/python       current.py   exit code 0: MIT OR Apache-2.0


Two environments, two versions of `packaging`, and both programs run, each with the Python of its own
environment. The notebook's own Python is a third environment, and nothing installed here changed it:
the tutorial still fails there, exactly as before.

### What activating changes

In a terminal, `source old-tutorial/bin/activate` saves typing `old-tutorial/bin/` before every
command. `run` gives a command a new shell of its own, so the activation and the commands it affects
go in one line, joined by `&&`:


In [10]:
code, printed = run("bash", "-c", 'source old-tutorial/bin/activate && echo "first on PATH: ${PATH%%:*}" '
                                  "&& command -v python && python tutorial.py")
print(printed)

activate = (PROJECT / "old-tutorial" / "bin" / "activate").read_text()
print("\nthe activate script holds the environment's full path:", str((PROJECT / "old-tutorial").resolve()) in activate)


first on PATH: old-tutorial/bin
old-tutorial/bin/python
['latest', '2.9', '2.10rc1', '2.10']

the activate script holds the environment's full path: True


After `activate`, the environment's `bin` came first on `PATH`, so typing `python` found the
environment's `python`, and the tutorial ran with `packaging` 21.3. That is all activating does. It
changes one shell, for as long as that shell runs, so in a notebook, where every command runs in a
new shell, an activation does not reach the next cell. `activate` also holds the environment's full
path, written in when the environment was made, which is one reason an environment cannot move.

### --system-site-packages

`--system-site-packages` makes an environment that also sees the packages of the Python it came from.
The notebook's Python has pytest, and `old-tutorial` does not:


In [11]:
run(sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", "shared")

for environment in ["old-tutorial", "shared"]:
    code, printed = run(f"{environment}/bin/python", "-c", "import pytest; print('pytest', pytest.__version__)")
    print(f"{environment:<13} {printed.splitlines()[-1]}")


old-tutorial  ModuleNotFoundError: No module named 'pytest'
shared        pytest 8.4.2


`shared` found pytest without anything being installed into it, because its `pyvenv.cfg` says
`include-system-site-packages = true`. That saves installing a large package twice, and it gives
back part of what an environment is for: what `shared` can import depends on what the Python outside
it holds, so a program that works in `shared` can fail in an environment with only its own packages.

### A project's environment, from nothing to its tests passing inside it

The pieces of this notebook, in one project. It gets an environment called `.venv`, as projects
usually name theirs, with pytest installed into it, and its tests run with the environment's Python.
One of the tests checks that it runs inside an environment:


In [12]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/readings.py


In [13]:
%%writefile scratch/stations/tests/test_readings.py
import sys

from readings import mean, to_fahrenheit


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


def test_mean_of_no_readings_is_none():
    assert mean([None, None]) is None


def test_boiling_point_in_fahrenheit():
    assert to_fahrenheit(100) == 212


def test_the_tests_run_inside_an_environment():
    assert sys.prefix != sys.base_prefix


Writing scratch/stations/tests/test_readings.py


In [14]:
steps = [
    ("make the environment", [sys.executable, "-m", "venv", "--without-pip", ".venv"]),
    ("install pytest into it", [sys.executable, "-m", "pip", "--python", ".venv", "--disable-pip-version-check",
                                "install", "-q", "pytest==8.4.2"]),
    ("check its pytest", [".venv/bin/python", "-m", "pytest", "--version"]),
    ("run the tests with it", [".venv/bin/python", "-m", "pytest", "-q", "--no-header"]),
]
for name, command in steps:
    code, printed = run(*command)
    print(f"--- {name}: exit code {code}")
    if printed:
        print(printed)


--- make the environment: exit code 0
--- install pytest into it: exit code 0
--- check its pytest: exit code 0
pytest 8.4.2
--- run the tests with it: exit code 0
....                                                                     [100%]
4 passed


Four commands, from an empty folder to a passing suite. The environment's `python -m pytest` ran the
tests, so the check that they ran inside an environment passed; run by the notebook's Python, that
test would fail. The environment holds pytest and the packages pytest needs, and nothing else.

### Where each part came from

| In the project | What it relies on | The section that showed it |
|---|---|---|
| `python -m venv --without-pip .venv` | a folder that works as a Python installation of its own | A new environment, and what is inside it |
| `--python .venv` | the notebook's own pip, installing for another Python | Packages installed with the notebook's own pip |
| `.venv/bin/python -m pytest` | the `python` you run decides the environment | The environment's python, run without activating anything |
| `test_the_tests_run_inside_an_environment` | `sys.prefix` differs from `sys.base_prefix` only inside one | The environment's python, run without activating anything |
| pytest in `.venv`, not borrowed from outside | an environment sees only its own packages | --system-site-packages |
| no `activate` anywhere | activation changes a shell, and nothing else | What activating changes |

`.venv` is a folder the project can delete and rebuild with the same four commands, which is how
Python's documentation expects an environment to be treated.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/07-virtual-environments-solutions.ipynb).

**1.** Make an environment called `task-env` in the project's folder, with `--without-pip`, and print
the names of the things in its folder and the keys in its `pyvenv.cfg`.


In [15]:
# your code here


**2.** Run `task-env/bin/python` to print whether it runs inside an environment.


In [16]:
# your code here


**3.** Install `packaging==21.3` and `pyparsing==3.3.2` into `task-env` with `pip`, and print what
`freeze` lists.


In [17]:
# your code here


**4.** Run `tutorial.py` with `task-env/bin/python` and with the notebook's Python, and print the
exit code and the last line of each.


In [18]:
# your code here


**5.** Make `task-shared` with `--without-pip` and `--system-site-packages`, and print whether
`import pytest` works in `task-env` and in `task-shared`.


In [19]:
# your code here


**6.** Remove `task-env` with `shutil.rmtree`, make it again, and print what `freeze` lists now.


In [20]:
# your code here


## Common errors

### No module named pip


In [21]:
code, printed = run("old-tutorial/bin/python", "-m", "pip", "install", "packaging==21.3")

print("exit code:", code)
print(printed)


exit code: 1
old-tutorial/bin/python: No module named pip


The environment's `python` found no module called `pip`, because the environment was made with
`--without-pip`. On your own computer, an environment made without that option has pip, and this
command works. Here, the notebook's pip installs for the environment's Python:


In [22]:
code, printed = pip("old-tutorial", "install", "-q", "packaging==21.3")

print("exit code:", code)


exit code: 0


### No module named pytest


In [23]:
run(sys.executable, "-m", "venv", "--without-pip", "fresh")
code, printed = run("fresh/bin/python", "-m", "pytest", "-q")

print("exit code:", code)
print(printed)


exit code: 1
fresh/bin/python: No module named pytest


pytest is in `.venv`, and in the notebook's own Python, and not in `fresh`, which sees only its own
packages. A package installed into one environment is not in any other, and the usual cause is an
install that went to a different environment from the one that runs the program. Install into the
environment that runs it:


In [24]:
code, printed = pip("fresh", "install", "-q", "pytest==8.4.2")
print("exit code:", code)

code, printed = run("fresh/bin/python", "-m", "pytest", "--version")
print(printed)


exit code: 0
pytest 8.4.2


### ModuleNotFoundError: No module named 'readings'


In [25]:
code, printed = run(".venv/bin/pytest", "-q", "--no-header")

print("exit code:", code)
print("\n".join(line for line in printed.splitlines() if line.startswith(("E ", "ERROR", "1 error"))))


exit code: 2
E   ModuleNotFoundError: No module named 'readings'
ERROR tests/test_readings.py
1 error


`.venv/bin/pytest` is the script pip installed with pytest, and it ran with the environment's
Python, yet the tests could not import `readings`. `python -m pytest` puts the folder it runs in on
the import path, as the **Your First Test** notebook said, and the `pytest` script does not, so a
module that sits in the project's folder, beside a `tests` folder, is found by one and not the
other. The cell printed only the `E` line and the counts, since the whole report shows Python's own
file paths. Run the environment's `python -m pytest`, as the project did, and the **Project Layout**
notebook shows how a project makes its code importable either way:


In [26]:
code, printed = run(".venv/bin/python", "-m", "pytest", "-q", "--no-header")

print("exit code:", code)
print(printed)


exit code: 0
....                                                                     [100%]
4 passed


### FileNotFoundError: [Errno 2] No such file or directory: 'moved/bin/pytest'


In [27]:
shutil.move(PROJECT / ".venv", PROJECT / "moved")

try:
    run("moved/bin/pytest", "--version")
except FileNotFoundError as error:
    print(f"{type(error).__name__}: {error}")

first_line = (PROJECT / "moved" / "bin" / "pytest").read_text().splitlines()[0]
print("the script's first line ends with:", first_line[-len(".venv/bin/python"):])
print("moved/bin/pytest exists:", (PROJECT / "moved" / "bin" / "pytest").exists())


FileNotFoundError: [Errno 2] No such file or directory: 'moved/bin/pytest'
the script's first line ends with: .venv/bin/python
moved/bin/pytest exists: True


The error says that `moved/bin/pytest` does not exist, and it does. The file that is missing is the
one named on the script's first line, the Python it must run with, which is still
`.venv/bin/python`, a path that went away with the move. An environment is not moved, as Python's
documentation says: it is removed and made again where it is needed, with the same commands, which
here take a second or two:


In [28]:
shutil.rmtree(PROJECT / "moved")
run(sys.executable, "-m", "venv", "--without-pip", ".venv")
pip(".venv", "install", "-q", "pytest==8.4.2")

code, printed = run(".venv/bin/python", "-m", "pytest", "-q", "--no-header")
print("exit code:", code)
print(printed)


exit code: 0
....                                                                     [100%]
4 passed


Last, the notebook is finished with its files, so this cell removes the scratch folder, with every
environment in it:


In [29]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A virtual environment is a folder with a `python`, its own `site-packages`, and a `pyvenv.cfg`
  that makes that `python` run inside the environment.
- Run the environment's `python` by its path, as scripts and notebooks do, and no activation is
  needed. Activating only puts the environment's `bin` first on one shell's `PATH`.
- `python -m pip --python ENVIRONMENT install ...` installs into an environment made with
  `--without-pip`, and `ENVIRONMENT/bin/python -m pip install ...` into one that has pip.
- One environment holds one version of each package, so programs that need different versions need
  different environments.
- An environment sees only its own packages, unless it was made with `--system-site-packages`.
- `python -m pytest` puts the project's folder on the import path, and the `pytest` script does not.
- An environment's scripts name its full path, so an environment is rebuilt where it is needed,
  never moved or copied, and never committed.


## What is next

The **Requirements and Pinning** notebook records what an environment holds, so that somebody else
can rebuild it exactly: `pip freeze`, requirements files, version specifiers, and the pins that make
an install give the same result next year.


---

&#8592; **Previous:** [Testing Failure](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/06-testing-failure.ipynb)  &nbsp;·&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)  &nbsp;·&nbsp;  **Next:** [Requirements and Pinning](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/08-requirements-and-pinning.ipynb) &#8594;
